In [ ]:
import os
import cv2
import pandas as pd
import numpy as np
import tensorflow as tf
from ultralytics import YOLO
from collections import defaultdict, deque
import time
import random

class OptimizedBirdPredictor:
    def __init__(self, yolo_model_path, tflite_model_path, conf_threshold=0.3):
        print("Initializing the optimized bird prediction system... 🚀")
        # 1. Load Object Detection & Tracking Model
        self.yolo_model = YOLO(yolo_model_path)
        print("✅ YOLOv8 model loaded.")

        # 2. Load Path Prediction Model (TFLite)
        try:
            self.interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
            self.interpreter.allocate_tensors()
            self.input_details = self.interpreter.get_input_details()
            self.output_details = self.interpreter.get_output_details()
            print("✅ TFLite LSTM model loaded.")
        except Exception as e:
            raise ValueError(f"Error loading TFLite model: {e}")

        # 3. Configuration & State Management
        self.conf_threshold = conf_threshold
        self.sequence_length = 15  # Must match the LSTM model's input size
        self.prediction_interval = 3  # Run prediction every 3 frames for a given bird
        self.track_timeout = 30  # Remove tracks not seen for 30 frames
        
        self.track_histories = defaultdict(lambda: deque(maxlen=self.sequence_length))
        self.track_predictions = {}
        self.track_colors = {}
        self.track_last_seen = {}
        self.all_detections_log = []
        
        self.processing_start_time = None
        self.first_prediction_time = None
        self.latency_measured = False

    def _get_track_color(self, track_id):
        """Assigns a unique, persistent color to each track ID."""
        if track_id not in self.track_colors:
            r, g, b = random.randint(50, 255), random.randint(50, 255), random.randint(50, 255)
            self.track_colors[track_id] = (b, g, r)
        return self.track_colors[track_id]
    
    def _cleanup_old_tracks(self, current_frame):
        """Remove tracks that haven't been seen recently."""
        tracks_to_remove = [
            track_id for track_id, last_seen in self.track_last_seen.items()
            if current_frame - last_seen > self.track_timeout
        ]
        for track_id in tracks_to_remove:
            self.track_histories.pop(track_id, None)
            self.track_predictions.pop(track_id, None)
            self.track_last_seen.pop(track_id, None)

    def process_video_stream(self, video_path, output_dir):
        """Processes a video file frame-by-frame, showing only predictions."""
        os.makedirs(output_dir, exist_ok=True)
        
        # Handle webcam input (integer 0) vs file path
        if isinstance(video_path, int):
            video_name = "webcam"
        else:
            video_name = os.path.basename(video_path)
        
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise ValueError(f"Error opening video source: {video_path}")

        frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = 0 if isinstance(video_path, int) else int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        # Only create output video if not using webcam
        if not isinstance(video_path, int):
            output_video_path = os.path.join(output_dir, f"{os.path.splitext(video_name)[0]}_COCO_predictions.mp4")
            out = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))
        else:
            out = None
            print("📹 Webcam mode: Press 'q' to quit")

        frame_number = 0
        print("\n--- Starting processing with YOLOv8m COCO Model ---")
        self.processing_start_time = time.time()
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break

            self._cleanup_old_tracks(frame_number)
            display_frame = frame.copy()
            
            # Specify `classes=[14]` to only track birds (COCO class 14 = bird)
            results = self.yolo_model.track(
                frame, persist=True, conf=self.conf_threshold, verbose=False, classes=[14]
            )
            
            if results[0].boxes is not None and results[0].boxes.id is not None:
                track_ids = results[0].boxes.id.int().cpu().tolist()
                boxes = results[0].boxes.xyxy.cpu()

                for i, track_id in enumerate(track_ids):
                    box = boxes[i]
                    center_x, center_y = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2
                    current_position = np.array([center_x, center_y])
                    
                    self.track_histories[track_id].append(current_position)
                    self.track_last_seen[track_id] = frame_number
                    self.all_detections_log.append({
                        "frame": frame_number, "track_id": track_id, "x1": float(box[0]), 
                        "y1": float(box[1]), "x2": float(box[2]), "y2": float(box[3]),
                        "center_x": float(center_x), "center_y": float(center_y)
                    })
                    
                    history = self.track_histories[track_id]
                    if len(history) == self.sequence_length and frame_number % self.prediction_interval == 0:
                        input_sequence = np.array(history, dtype=np.float32).reshape(1, self.sequence_length, 2)
                        
                        self.interpreter.set_tensor(self.input_details[0]['index'], input_sequence)
                        self.interpreter.invoke()
                        predicted_path = self.interpreter.get_tensor(self.output_details[0]['index'])
                        
                        current_pos_int = tuple(current_position.astype(int))
                        predicted_next_point = tuple(predicted_path[0, 0, :].astype(int))
                        self.track_predictions[track_id] = (current_pos_int, predicted_next_point)
                        
                        if not self.latency_measured:
                            self.first_prediction_time = time.time()
                            latency = self.first_prediction_time - self.processing_start_time
                            print(f"⏱️  First prediction latency: {latency:.3f} seconds")
                            self.latency_measured = True

            for track_id, (start_point, end_point) in self.track_predictions.items():
                if self.track_last_seen.get(track_id) == frame_number:
                    color = self._get_track_color(track_id)
                    cv2.arrowedLine(display_frame, start_point, end_point, color, 3, tipLength=0.4)
                    cv2.circle(display_frame, start_point, 5, color, -1)
                    cv2.putText(display_frame, f"ID:{track_id}", (start_point[0] + 10, start_point[1] - 10), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            
            # Write to output video if not using webcam
            if out is not None:
                out.write(display_frame)
            
            # Show frame in window
            cv2.imshow('Bird Tracking', display_frame)
            
            # Check for quit key (q)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
                
            if frame_number > 0 and frame_number % 100 == 0 and not isinstance(video_path, int):
                print(f"  Processed frame {frame_number}/{total_frames} ({(frame_number/total_frames)*100:.1f}%)")
            frame_number += 1
            
        cap.release()
        if out is not None:
            out.release()
        cv2.destroyAllWindows()
        
        print("\n--- Processing Complete ---")
        
        # Only save files if not using webcam
        if not isinstance(video_path, int):
            print(f"✅ Output video saved to: {output_video_path}")
            csv_path = os.path.join(output_dir, f"{os.path.splitext(video_name)[0]}_COCO_detections_log.csv")
            pd.DataFrame(self.all_detections_log).to_csv(csv_path, index=False)
            print(f"✅ Detection log saved to: {csv_path}")
        
        unique_birds = len(set(d['track_id'] for d in self.all_detections_log)) if self.all_detections_log else 0
        print(f"✅ Tracked {unique_birds} unique birds.")
        print(f"✅ Processed {frame_number} frames total.")


if __name__ == "__main__":
    # --- Configuration ---
    YOLO_MODEL_PATH = "yolov8x.pt"  # This will be downloaded automatically if not found
    TFLITE_MODEL_PATH = "D:/Project_P1/results/video_analysis/model.tflite"
    
    # Choose one of these options:
    VIDEO_FILE_PATH = "D:/Project_P1/data/valid/test1.mp4"  # For video file
    # VIDEO_FILE_PATH = 0  # For webcam input
    
    VIDEO_OUTPUT_DIR = "D:/Project_P1/results/coco_model_analysis"

    # --- Initialize and Run ---
    predictor = OptimizedBirdPredictor(YOLO_MODEL_PATH, TFLITE_MODEL_PATH, conf_threshold=0.3)
    predictor.process_video_stream(VIDEO_FILE_PATH, VIDEO_OUTPUT_DIR)

Initializing the optimized bird prediction system... 🚀
✅ YOLOv8 model loaded.
✅ TFLite LSTM model loaded.
📹 Webcam mode: Press 'q' to quit

--- Starting processing with YOLOv8m COCO Model ---
WARNING not enough matching points
⏱️  First prediction latency: 139.770 seconds

--- Processing Complete ---
✅ Tracked 1 unique birds.
✅ Processed 1452 frames total.


In [1]:
import os
import cv2
import numpy as np
import tensorflow as tf
from ultralytics import YOLO
from collections import defaultdict
import random
import time
import threading
import queue

# =============================================================================
# --- Configuration & Parameters ---
# =============================================================================

PROCESS_EVERY_N_FRAMES = 1  # Process every frame for minimal lag

SEQUENCE_LENGTH = 15
PRED_EVERY_N_FRAMES = 1  # Run prediction every frame (for smoothness)

FONT = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 0.5
BOX_THICKNESS = 1
PREDICTION_THICKNESS = 2
DOT_RADIUS = 3
TEXT_THICKNESS = 1
COLOR_PALETTE_SIZE = 50

MAX_QUEUE_SIZE = 3  # Reduce queue to avoid backlog
INFERENCE_BATCH_SIZE = 1

INTERPOLATE_SKIPPED_FRAMES = True  # Optional linear interpolation for skipped frames

# =============================================================================
# --- Helper Class: TrackBuffer ---
# =============================================================================
class TrackBuffer:
    def __init__(self, seq_len):
        self.seq_len = seq_len
        self.buf = np.zeros((seq_len, 2), dtype=np.float32)
        self.ptr = 0
        self.count = 0

    def append(self, pos):
        self.buf[self.ptr] = pos
        self.ptr = (self.ptr + 1) % self.seq_len
        if self.count < self.seq_len:
            self.count += 1

    def is_full(self):
        return self.count == self.seq_len

    def get_ordered(self):
        if self.count < self.seq_len:
            return self.buf[:self.count].copy()
        return np.roll(self.buf, -self.ptr, axis=0)

# =============================================================================
# --- Threaded Inference Class ---
# =============================================================================
class YOLOInferenceThread(threading.Thread):
    def __init__(self, yolo_model, conf_threshold):
        super().__init__()
        self.yolo_model = yolo_model
        self.conf_threshold = conf_threshold
        self.input_queue = queue.Queue(maxsize=MAX_QUEUE_SIZE)
        self.output_queue = queue.Queue(maxsize=MAX_QUEUE_SIZE)
        self.running = True
        self.daemon = True

    def run(self):
        while self.running:
            try:
                frame_data = self.input_queue.get(timeout=0.1)
                frame, frame_number = frame_data

                results = self.yolo_model.track(
                    frame,
                    persist=True,
                    conf=self.conf_threshold,
                    imgsz=640,
                    verbose=False,
                    device='0'
                )
                self.output_queue.put((results[0], frame_number))

            except queue.Empty:
                continue
            except Exception as e:
                print(f"Inference error: {e}")
                continue

    def stop(self):
        self.running = False

# =============================================================================
# --- Main Class: RealTimeBirdPredictor ---
# =============================================================================
class RealTimeBirdPredictor:
    def __init__(self, yolo_path, tflite_path, conf_threshold=0.3):
        print("Initializing the real-time bird prediction system... 🚀")

        self.yolo_model = YOLO(yolo_path)
        self.yolo_model.to("cuda")  # FP16 for speed

        self.inference_thread = YOLOInferenceThread(self.yolo_model, conf_threshold)
        self.inference_thread.start()
        print("✅ YOLO inference thread started.")

        try:
            self.interpreter = tf.lite.Interpreter(model_path=tflite_path)
            self.interpreter.allocate_tensors()
            self.input_details = self.interpreter.get_input_details()
            self.output_details = self.interpreter.get_output_details()
            self.input_index = self.input_details[0]['index']
            self.output_index = self.output_details[0]['index']
            print("✅ TFLite LSTM model loaded.")
        except Exception as e:
            raise ValueError(f"Error loading TFLite model: {e}")

        self.conf_threshold = conf_threshold
        self.track_buffers = defaultdict(lambda: TrackBuffer(SEQUENCE_LENGTH))
        self.track_predictions = {}
        self.track_colors = {}
        self.track_first_seen = {}
        self.color_palette = self._generate_colors(COLOR_PALETTE_SIZE)
        self.frame_times = []
        self.prediction_times = []
        self.last_results = None
        self.skipped_frames = 0
        self.processed_frames = 0

    def _generate_colors(self, num_colors):
        rng = random.Random(42)
        return [tuple(rng.randint(60, 255) for _ in range(3)) for _ in range(num_colors)]

    def _draw_prediction(self, frame, current_pos, predicted_points, color):
        if predicted_points is None or len(predicted_points) == 0:
            return
        all_points = [tuple(map(int, current_pos))] + [tuple(map(int, p)) for p in predicted_points]
        for i in range(len(all_points) - 1):
            cv2.line(frame, all_points[i], all_points[i+1], color, PREDICTION_THICKNESS)
        cv2.circle(frame, all_points[0], DOT_RADIUS + 1, color, -1)

    def _interpolate_positions(self, pos1, pos2, alpha):
        return pos1 * (1 - alpha) + pos2 * alpha

    def process_stream(self, source, output_path=None):
        is_webcam = str(source).lower() == 'webcam'
        cap_source = 0 if is_webcam else source
        cap = cv2.VideoCapture(cap_source)
        if not cap.isOpened():
            raise IOError(f"Cannot open video source: {source}")

        frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        # --- Auto-detect FPS ---
        if is_webcam:
            TARGET_FPS = 30
        else:
            TARGET_FPS = cap.get(cv2.CAP_PROP_FPS)
            if TARGET_FPS <= 0 or TARGET_FPS > 120:
                TARGET_FPS = 30
        FRAME_INTERVAL = 1.0 / TARGET_FPS
        fps = TARGET_FPS

        out = None
        if output_path:
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))
            print(f"🎥 Output will be saved to: {output_path}")

        frame_number = 0
        last_processed_frame = -1

        print(f"Processing with frame skipping: 1 of every {PROCESS_EVERY_N_FRAMES} frames processed by YOLO")
        print(f"Skipping {PROCESS_EVERY_N_FRAMES - 1} frames between YOLO processing")

        while True:
            t_start = time.perf_counter()
            ret, frame = cap.read()
            if not ret:
                print("End of video stream.")
                break

            active_track_ids = set()

            # --- Submit frame to YOLO thread ---
            if frame_number % PROCESS_EVERY_N_FRAMES == 0:
                if self.inference_thread.input_queue.qsize() < MAX_QUEUE_SIZE:
                    self.inference_thread.input_queue.put((frame, frame_number))
                    self.processed_frames += 1
                else:
                    self.skipped_frames += 1

            # --- Poll latest inference result ---
            latest_results = None
            while not self.inference_thread.output_queue.empty():
                latest_results, result_frame_num = self.inference_thread.output_queue.get_nowait()
            if latest_results is not None:
                self.last_results = latest_results
                last_processed_frame = result_frame_num

            current_results = self.last_results

            # --- Process detections ---
            if current_results is not None and current_results.boxes is not None and current_results.boxes.id is not None:
                track_ids = current_results.boxes.id.int().cpu().tolist()
                boxes = current_results.boxes.xyxy.cpu()
                active_track_ids.update(track_ids)

                for i, track_id in enumerate(track_ids):
                    box = boxes[i]
                    center_x = (box[0] + box[2]) / 2.0
                    center_y = (box[1] + box[3]) / 2.0
                    current_position = np.array([center_x, center_y], dtype=np.float32)

                    if track_id not in self.track_first_seen:
                        self.track_first_seen[track_id] = frame_number

                    # Optional interpolation for skipped frames
                    if INTERPOLATE_SKIPPED_FRAMES and track_id in self.track_buffers:
                        buffer = self.track_buffers[track_id]
                        if buffer.count > 0:
                            prev_pos = buffer.buf[(buffer.ptr - 1) % buffer.seq_len]
                            current_position = self._interpolate_positions(prev_pos, current_position, 0.5)

                    self.track_buffers[track_id].append(current_position)

                    buffer = self.track_buffers[track_id]
                    if buffer.is_full() and (frame_number - self.track_first_seen[track_id]) % PRED_EVERY_N_FRAMES == 0:
                        history = buffer.get_ordered()
                        normalized_history = history / np.array([frame_width, frame_height], dtype=np.float32)
                        input_tensor = np.expand_dims(normalized_history, axis=0).astype(np.float32)

                        t_pred_start = time.perf_counter()
                        self.interpreter.set_tensor(self.input_index, input_tensor)
                        self.interpreter.invoke()
                        pred_normalized = self.interpreter.get_tensor(self.output_index)
                        self.prediction_times.append((time.perf_counter() - t_pred_start) * 1000)

                        pred_path = pred_normalized[0] * np.array([frame_width, frame_height], dtype=np.float32)
                        self.track_predictions[track_id] = pred_path

            # --- Visualization ---
            if current_results is not None and current_results.boxes is not None and current_results.boxes.id is not None:
                track_ids_to_draw = current_results.boxes.id.int().cpu().tolist()
                boxes_to_draw = current_results.boxes.xyxy.cpu()

                for i, track_id in enumerate(track_ids_to_draw):
                    box = boxes_to_draw[i]
                    color = self.track_colors.setdefault(track_id, self.color_palette[len(self.track_colors) % COLOR_PALETTE_SIZE])
                    cv2.rectangle(frame, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), color, BOX_THICKNESS)

                    if track_id in self.track_predictions:
                        history = self.track_buffers[track_id].get_ordered()
                        if history.shape[0] > 0:
                            self._draw_prediction(frame, history[-1], self.track_predictions[track_id], color)
                            last_pos = history[-1]
                            label = f"ID:{track_id}"
                            cv2.putText(frame, label, (int(last_pos[0]) + 7, int(last_pos[1]) - 7), FONT, FONT_SCALE, color, TEXT_THICKNESS)

            t_end = time.perf_counter()
            frame_time_ms = (t_end - t_start) * 1000
            self.frame_times.append(frame_time_ms)

            window_size = min(30, len(self.frame_times))
            avg_frame_time = np.mean(self.frame_times[-window_size:]) if self.frame_times else 0
            avg_pred_time = np.mean(self.prediction_times[-window_size:]) if self.prediction_times else 0
            current_fps = 1000.0 / avg_frame_time if avg_frame_time > 0 else 0

            cv2.putText(frame, f"FPS: {current_fps:.1f}", (10, 20), FONT, FONT_SCALE, (0, 255, 0), TEXT_THICKNESS)
            cv2.putText(frame, f"Frame: {frame_number} (YOLO: {self.processed_frames})", (10, 40), FONT, FONT_SCALE, (0, 255, 0), TEXT_THICKNESS)
            cv2.putText(frame, f"Skipped: {self.skipped_frames}", (10, 60), FONT, FONT_SCALE, (0, 255, 0), TEXT_THICKNESS)

            cv2.imshow("Real-Time Bird Prediction", frame)
            if out:
                out.write(frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                print("Interrupted by user.")
                break

            frame_number += 1
            # Sync display to video FPS
            time_elapsed = time.perf_counter() - t_start
            time_to_wait = FRAME_INTERVAL - time_elapsed
            if time_to_wait > 0:
                time.sleep(time_to_wait)

        # Cleanup
        self.inference_thread.stop()
        self.inference_thread.join()
        cap.release()
        if out:
            out.release()
        cv2.destroyAllWindows()

        print("\n--- Processing Complete ---")
        if self.frame_times:
            avg_total_fps = 1000 / np.mean(self.frame_times)
            print(f"Average processing FPS: {avg_total_fps:.2f}")
            print(f"Average frame processing time: {np.mean(self.frame_times):.2f} ms")
            print(f"YOLO frames processed: {self.processed_frames}")
            print(f"Frames skipped in queue: {self.skipped_frames}")
        if self.prediction_times:
            print(f"Average TFLite prediction time: {np.mean(self.prediction_times):.2f} ms")

# =============================================================================
# --- Script Entry Point ---
# =============================================================================
if __name__ == "__main__":
    SOURCE_PATH = "D:/Project_P1/data/valid/tt.mp4"
    YOLO_MODEL_PATH = "D:/Project_P1/src/ml/runs/detect/train2/weights/best.pt"
    TFLITE_MODEL_PATH = "D:/Project_P1/results/video_analysis/model.tflite"
    OUTPUT_FILE_PATH = "D:/Project_P1/results/optimized_output.mp4"
    CONFIDENCE_THRESHOLD = 0.3

    try:
        predictor = RealTimeBirdPredictor(
            yolo_path=YOLO_MODEL_PATH,
            tflite_path=TFLITE_MODEL_PATH,
            conf_threshold=CONFIDENCE_THRESHOLD
        )
        predictor.process_stream(
            source=SOURCE_PATH,
            output_path=OUTPUT_FILE_PATH
        )
    except Exception as e:
        print(f"An error occurred: {e}")


Initializing the real-time bird prediction system... 🚀
✅ YOLO inference thread started.
✅ TFLite LSTM model loaded.
🎥 Output will be saved to: D:/Project_P1/results/optimized_output.mp4
Processing with frame skipping: 1 of every 1 frames processed by YOLO
Skipping 0 frames between YOLO processing
End of video stream.

--- Processing Complete ---
Average processing FPS: 222.90
Average frame processing time: 4.49 ms
YOLO frames processed: 193
Frames skipped in queue: 29
Average TFLite prediction time: 1.33 ms


## Review - 03

In [2]:
import os
import cv2
import numpy as np
import tensorflow as tf
from ultralytics import YOLO
from collections import defaultdict
import random
import time
import threading
import queue

# =============================================================================
# --- Configuration & Parameters ---
# =============================================================================

PROCESS_EVERY_N_FRAMES = 1  # Process every frame for minimal lag

SEQUENCE_LENGTH = 15
PRED_EVERY_N_FRAMES = 1  # Run prediction every frame (for smoothness)

FONT = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 0.5
BOX_THICKNESS = 1
PREDICTION_THICKNESS = 2
DOT_RADIUS = 3
TEXT_THICKNESS = 1
COLOR_PALETTE_SIZE = 50

MAX_QUEUE_SIZE = 3  # Reduce queue to avoid backlog
INFERENCE_BATCH_SIZE = 1

# =============================================================================
# --- Helper Class: TrackBuffer ---
# =============================================================================
class TrackBuffer:
    """A fixed-size ring buffer to efficiently store position history for each track."""
    def __init__(self, seq_len):
        self.seq_len = seq_len
        self.buf = np.zeros((seq_len, 2), dtype=np.float32)
        self.ptr = 0
        self.count = 0

    def append(self, pos):
        self.buf[self.ptr] = pos
        self.ptr = (self.ptr + 1) % self.seq_len
        if self.count < self.seq_len:
            self.count += 1

    def is_full(self):
        return self.count == self.seq_len

    def get_ordered(self):
        if self.count < self.seq_len:
            return self.buf[:self.count].copy()
        return np.roll(self.buf, -self.ptr, axis=0)

# =============================================================================
# --- Threaded Inference Class ---
# =============================================================================
class YOLOInferenceThread(threading.Thread):
    """A separate thread for running YOLO inference to prevent blocking the main thread."""
    def __init__(self, yolo_model, conf_threshold):
        super().__init__()
        self.yolo_model = yolo_model
        self.conf_threshold = conf_threshold
        self.input_queue = queue.Queue(maxsize=MAX_QUEUE_SIZE)
        self.output_queue = queue.Queue(maxsize=MAX_QUEUE_SIZE)
        self.running = True
        self.daemon = True

    def run(self):
        while self.running:
            try:
                frame_data = self.input_queue.get(timeout=0.1)
                frame, frame_number = frame_data

                results = self.yolo_model.track(
                    frame,
                    persist=True,
                    conf=self.conf_threshold,
                    imgsz=640,
                    verbose=False,
                    device='0'
                )
                self.output_queue.put((results[0], frame_number))

            except queue.Empty:
                continue
            except Exception as e:
                print(f"Inference error: {e}")
                continue

    def stop(self):
        self.running = False

# =============================================================================
# --- Main Class: RealTimeBirdPredictor ---
# =============================================================================
class RealTimeBirdPredictor:
    def __init__(self, yolo_path, tflite_path, conf_threshold=0.3):
        print("Initializing the real-time bird prediction system... 🚀")

        self.yolo_model = YOLO(yolo_path)
        self.yolo_model.to("cuda")

        self.inference_thread = YOLOInferenceThread(self.yolo_model, conf_threshold)
        self.inference_thread.start()
        print("✅ YOLO inference thread started.")

        try:
            self.interpreter = tf.lite.Interpreter(model_path=tflite_path)
            self.interpreter.allocate_tensors()
            self.input_details = self.interpreter.get_input_details()
            self.output_details = self.interpreter.get_output_details()
            self.input_index = self.input_details[0]['index']
            self.output_index = self.output_details[0]['index']
            print("✅ TFLite LSTM model loaded.")
        except Exception as e:
            raise ValueError(f"Error loading TFLite model: {e}")

        self.conf_threshold = conf_threshold
        self.track_buffers = defaultdict(lambda: TrackBuffer(SEQUENCE_LENGTH))
        self.track_predictions = {}
        self.track_colors = {}
        self.track_first_seen = {}
        self.color_palette = self._generate_colors(COLOR_PALETTE_SIZE)
        self.frame_times = []
        self.prediction_times = []
        self.last_results = None
        self.skipped_frames = 0
        self.processed_frames = 0
        
        # Heatmap attributes
        self.flight_path_heatmap = None
        self.flight_path_risk_map = None

    def _generate_colors(self, num_colors):
        rng = random.Random(42)
        return [tuple(rng.randint(60, 255) for _ in range(3)) for _ in range(num_colors)]

    def _create_flight_path_heatmap(self, H, W):
        """Pre-calculates the heatmap for the flight path."""
        print("Creating flight path risk heatmap...")
        risk_map = np.zeros((H, W), dtype=np.uint8)
        
        start_point = (0, H - 1)
        end_point = (W - 1, 0)
        
        cv2.line(risk_map, start_point, end_point, color=255, thickness=150)
        
        risk_map = cv2.GaussianBlur(risk_map, (0, 0), sigmaX=80, sigmaY=80)

        heatmap = cv2.applyColorMap(risk_map, cv2.COLORMAP_JET)
        
        self.flight_path_risk_map = risk_map
        self.flight_path_heatmap = heatmap
        print("✅ Heatmap created.")

    def _draw_prediction(self, frame, current_pos, predicted_points, color):
        if predicted_points is None or len(predicted_points) == 0:
            return
        all_points = [tuple(map(int, current_pos))] + [tuple(map(int, p)) for p in predicted_points]
        for i in range(len(all_points) - 1):
            cv2.line(frame, all_points[i], all_points[i+1], color, PREDICTION_THICKNESS)
        cv2.circle(frame, all_points[0], DOT_RADIUS + 1, color, -1)

    def process_stream(self, source, output_path=None):
        cap = cv2.VideoCapture(source)
        if not cap.isOpened():
            raise IOError(f"Cannot open video source: {source}")

        frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        
        # Corrected line
        self._create_flight_path_heatmap(frame_height, frame_width)

        TARGET_FPS = cap.get(cv2.CAP_PROP_FPS)
        if TARGET_FPS <= 0 or TARGET_FPS > 120: TARGET_FPS = 30
        FRAME_INTERVAL = 1.0 / TARGET_FPS
        
        out = None
        if output_path:
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            out = cv2.VideoWriter(output_path, fourcc, TARGET_FPS, (frame_width, frame_height))
            print(f"🎥 Output will be saved to: {output_path}")

        frame_number = 0
        while True:
            t_start = time.perf_counter()
            ret, frame = cap.read()
            if not ret:
                print("End of video stream.")
                break

            # --- Apply the heatmap to the frame ---
            display_frame = cv2.addWeighted(frame, 0.6, self.flight_path_heatmap, 0.4, 0)
            
            # --- MODIFICATION: Track max risk in the current frame ---
            max_risk_in_frame = 0

            if frame_number % PROCESS_EVERY_N_FRAMES == 0:
                if self.inference_thread.input_queue.qsize() < MAX_QUEUE_SIZE:
                    self.inference_thread.input_queue.put((frame, frame_number))
                    self.processed_frames += 1
                else:
                    self.skipped_frames += 1

            latest_results = None
            while not self.inference_thread.output_queue.empty():
                latest_results, _ = self.inference_thread.output_queue.get_nowait()
            if latest_results is not None:
                self.last_results = latest_results
            
            current_results = self.last_results

            if current_results and current_results.boxes and current_results.boxes.id is not None:
                track_ids = current_results.boxes.id.int().cpu().tolist()
                boxes = current_results.boxes.xyxy.cpu()

                for i, track_id in enumerate(track_ids):
                    box = boxes[i]
                    center_x = (box[0] + box[2]) / 2.0
                    center_y = (box[1] + box[3]) / 2.0
                    current_position = np.array([center_x, center_y], dtype=np.float32)

                    if track_id not in self.track_first_seen:
                        self.track_first_seen[track_id] = frame_number
                    self.track_buffers[track_id].append(current_position)

                    buffer = self.track_buffers[track_id]
                    if buffer.is_full() and (frame_number - self.track_first_seen.get(track_id, 0)) % PRED_EVERY_N_FRAMES == 0:
                        history = buffer.get_ordered()
                        normalized_history = history / np.array([frame_width, frame_height], dtype=np.float32)
                        input_tensor = np.expand_dims(normalized_history, axis=0).astype(np.float32)
                        
                        self.interpreter.set_tensor(self.input_index, input_tensor)
                        self.interpreter.invoke()
                        pred_normalized = self.interpreter.get_tensor(self.output_index)
                        
                        pred_path = pred_normalized[0] * np.array([frame_width, frame_height], dtype=np.float32)
                        self.track_predictions[track_id] = pred_path
                    
                    # --- Visualization and Risk Calculation ---
                    pos_x, pos_y = int(current_position[0]), int(current_position[1])
                    risk_score = 0
                    if 0 <= pos_y < frame_height and 0 <= pos_x < frame_width:
                        risk_score = int(self.flight_path_risk_map[pos_y, pos_x])
                    
                    # --- MODIFICATION: Update the maximum risk for the frame ---
                    max_risk_in_frame = max(max_risk_in_frame, risk_score)

                    risk_color = (0, 255, 0)
                    if risk_score > 100: risk_color = (0, 255, 255)
                    if risk_score > 180: risk_color = (0, 0, 255)
                    
                    label = f"ID:{track_id} Risk:{risk_score}"
                    
                    if track_id in self.track_predictions:
                        history = self.track_buffers[track_id].get_ordered()
                        if history.shape[0] > 0:
                            self._draw_prediction(display_frame, history[-1], self.track_predictions[track_id], risk_color)
                            last_pos = history[-1]
                            cv2.putText(display_frame, label, (int(last_pos[0]) + 7, int(last_pos[1]) - 7), FONT, FONT_SCALE, risk_color, TEXT_THICKNESS)

            # --- MODIFICATION: Display overall risk level instead of frame number ---
            overall_risk_text = f"OVERALL RISK LEVEL: {max_risk_in_frame}"
            overall_risk_color = (0, 255, 0)  # Green
            if max_risk_in_frame > 100: overall_risk_color = (0, 255, 255) # Yellow
            if max_risk_in_frame > 180: overall_risk_color = (0, 0, 255)   # Red
            
            cv2.putText(display_frame, overall_risk_text, (10, 30), FONT, 0.7, overall_risk_color, 2)
            
            cv2.imshow("Real-Time Bird Prediction with Risk Analysis", display_frame)
            if out:
                out.write(display_frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

            frame_number += 1
            time_elapsed = time.perf_counter() - t_start
            time_to_wait = FRAME_INTERVAL - time_elapsed
            if time_to_wait > 0:
                time.sleep(time_to_wait)

        # Cleanup
        self.inference_thread.stop()
        self.inference_thread.join()
        cap.release()
        if out: out.release()
        cv2.destroyAllWindows()
        print("\n--- Processing Complete ---")

# =============================================================================
# --- Script Entry Point ---
# =============================================================================
if __name__ == "__main__":
    SOURCE_PATH = "D:/Project_P1/data/valid/test1.mp4"
    YOLO_MODEL_PATH = "D:/Project_P1/src/ml/runs/detect/train2/weights/best.pt"
    TFLITE_MODEL_PATH = "D:/Project_P1/results/video_analysis/model.tflite"
    OUTPUT_FILE_PATH = "D:/Project_P1/results/risk_analysis_output.mp4"
    CONFIDENCE_THRESHOLD = 0.3

    try:
        predictor = RealTimeBirdPredictor(
            yolo_path=YOLO_MODEL_PATH,
            tflite_path=TFLITE_MODEL_PATH,
            conf_threshold=CONFIDENCE_THRESHOLD
        )
        predictor.process_stream(
            source=SOURCE_PATH,
            output_path=OUTPUT_FILE_PATH
        )
    except Exception as e:
        print(f"An error occurred: {e}")

Initializing the real-time bird prediction system... 🚀
✅ YOLO inference thread started.
✅ TFLite LSTM model loaded.
Creating flight path risk heatmap...
✅ Heatmap created.
🎥 Output will be saved to: D:/Project_P1/results/risk_analysis_output.mp4
End of video stream.

--- Processing Complete ---
